# Credit Risk Modelling in Python

Credit risk is the possibility that a borrower will not be able to make timely payments and will default on their debt. It refers to the possibility that a lender may not get the interest or principle due to them on time.

Financial organisations are concerned about reducing the risk of default. As a result, commercial and investment banks, venture capital funds, asset management organisations, and insurance corporations, to mention a few, are increasingly depending on technology to anticipate which customers are most likely to default on their obligations.

#### Machine Learning models have helped these companies in improving the accuracy of their credit risk analysis by giving a scientific technique for predicting potential debtors.

##### The objective of this article is to build a model to predict probability of person defaulting a loan. The following steps will be followed in building the model.

- Data Preprocessing and data prepration
- Features Engineering and Features selection
- Model Developement and Model Evaluations

Data links:
https://www.kaggle.com/uciml/default-of-credit-card-clients-dataset


## Importing the libraries and reading the data

In [ ]:
# Importing libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

: 

In [ ]:
# Reading the data 
credit_risk= pd.read_csv("UCI_credit_card.csv")
credit_risk.head()

In [ ]:
### Here I am doing to copy the original data in data frame called df.
df= credit_risk.copy()
df.head()

## Data preparation and Pre-processing

The data set was taken from kaggle for credit risk modelling. The data initially revealed a total of 25 features. It is very important to clean the data in a suitable format before developing any machine learning model.

In [ ]:
# Lets see the information of data
df.info()

We are working with a data set containing 25 features for 30,000 clients. "default.payment.next.month" is a feature and is the target variable we are trying to predict.

In [ ]:
# As we seen Column ID has no meaning here so, we will remove it
df.drop(["ID"], axis=1, inplace= True) #axis=1 -- column removal and inplcae= True --means change in the original data

In [ ]:
# Lets check the statistics of data
df.describe()

In [ ]:
# checking for missing values
df.isnull().sum()

 We do not find any missing values in our data set.

### Analysis of column "EDUCATION"

EDUCATION: (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)

In [ ]:
# Lets see the value counts in EDUCATION columns:
df.EDUCATION.value_counts()

In [ ]:
# From the Data Description given, we know that in df.EDUCATION, 5 and 6 represents "unknown" 
#Changing 0,5 and 6 to keep it under 1 category.

df['EDUCATION'].replace({0:1,1:1,2:2,3:3,4:4,5:1,6:1}, inplace=True)
df.EDUCATION.value_counts()

### Aalysis of column "MARRIAGE"

Marital status (1=married, 2=single, 3=others)

In [ ]:
# lets see the values count in column marriage
df['MARRIAGE'].value_counts()

In [ ]:
# Here I am going to map 0 with 1
df['MARRIAGE'].replace({0:1,1:1,2:2,3:3}, inplace=True)
df['MARRIAGE'].value_counts()

### Analysis of column "PAY_0 to PAY_6"

PAY_0: Repayment status in September, 2005 (-1=pay duly, 1=payment delay for one month, 2=payment delay for two months, … 8=payment delay for eight months, 9=payment delay for nine months and above)

In [ ]:
# Lets see the value counts in column 'PAY_0'
df['PAY_0'].value_counts()

## Data Visualization

In [ ]:
# Lets visualize the target column "default.payment.next.month"
plt.figure(figsize=(6,6))
sns.countplot(df['default.payment.next.month'])
plt.xticks([0,1], labels=["Not Deafaulted", "Defaulted"])
plt.title("Target Distribution")

In [ ]:
# Lets count it
df['default.payment.next.month'].value_counts()

So, By looking at the target varibale we could say that, our data is pretty much imbalance. We would like to make it balanced before going to trained the model.

In [ ]:
# with age column
plt.figure(figsize=(6,6))
sns.displot(df['AGE'], kde=True)
plt.xticks(rotation=0)
plt.ylabel('Count')
plt.title("Age distribution")

In [ ]:
# With sex columns 
#Gender (1=male, 2=female)
plt.figure(figsize=(6,6))
sns.countplot('SEX',hue='default.payment.next.month',data=df)
plt.xticks([0,1], labels=["Male", "Female"])
#plt.xlabel({1:'Male',2:'Feamle'})
plt.title("Gender Distribution")

In [ ]:
# With EDUCATION columns 
# (1=graduate school, 2=university, 3=high school, 4=others, 5=unknown, 6=unknown)
plt.figure(figsize=(10,6))
sns.countplot('EDUCATION', hue='default.payment.next.month',data=df)
plt.xticks([0,1,2,3],labels=["graduate school", "university",'high school','others'])
plt.title("Education Distribution")

In [ ]:
# With MARRIAGE columns 
#
plt.figure(figsize=(10,6))
sns.countplot('MARRIAGE',hue='default.payment.next.month',data=df)
plt.xticks([0,1,2],labels=["Married", "single",'others'])
plt.title("MARRIAGE Distribution")

In [ ]:
# With column 'limit_bal'
sns.displot(df.LIMIT_BAL, kde=True)


In [ ]:
plt.subplots(figsize=(20,10))
plt.subplot(231)
plt.scatter(x=df.PAY_AMT1, y=df.BILL_AMT1, c='r', s=1)
plt.xlabel('PAY_AMT1')
plt.ylabel('BILL_AMT1')

plt.subplot(232)
plt.scatter(x=df.PAY_AMT2, y=df.BILL_AMT2, c='g', s=1)
plt.xlabel('PAY_AMT2')
plt.ylabel('BILL_AMT2')
plt.title('Payment structure vs Bill amount in the last 6 months', fontsize=15)

plt.subplot(233)
plt.scatter(x=df.PAY_AMT3, y=df.BILL_AMT3, c='b', s=1)
plt.xlabel('PAY_AMT3')
plt.ylabel('BILL_AMT3')

plt.subplot(234)
plt.scatter(x=df.PAY_AMT4, y=df.BILL_AMT4, c='y', s=1)
plt.xlabel('PAY_AMT4')
plt.ylabel('BILL_AMT4')

plt.subplot(235)
plt.scatter(x=df.PAY_AMT5, y=df.BILL_AMT5, c='black', s=1)
plt.xlabel('PAY_AMT5')
plt.ylabel('BILL_AMT5')

plt.subplot(236)
plt.scatter(x=df.PAY_AMT6, y=df.BILL_AMT6, c='m', s=1)
plt.xlabel('PAY_AMT6')
plt.ylabel('BILL_AMT6')


plt.show()

## Creating Independent features and depened features

In [ ]:
# Independnet features
X = df.drop(['default.payment.next.month'], axis=1)
# Dependent feature
y = df['default.payment.next.month']
X.head()

## Scalling the features
- So, scalling the independent features are very important so that our model is not biased toward the higher range of values. To make all features in same range
- Here we are using StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler= StandardScaler()
X= scaler.fit_transform(X)

## Train Test Split

In [ ]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.20,random_state=42)

## Balancing the data
- since our dataset was imbalace, so used SMOTE technique to make it balanced.
- For over-sampling techniques, SMOTE (Synthetic Minority Oversampling Technique) is considered as one of the most popular and influential data sampling algorithms in ML and data mining. With SMOTE, the minority class is over-sampled by creating “synthetic” examples rather than by over-sampling with replacement [2]. These introduced synthetic examples are based along the line segments joining a defined number of k minority class nearest neighbours, which is in the imblearn package is set at five by default.

In [ ]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# summarize class distribution
print("Before oversampling: ",Counter(y_train))

# define oversampling strategy
SMOTE= SMOTE()

# fit and apply the transform 
X_train,y_train= SMOTE.fit_resample(X_train,y_train)

# summarize class distribution
print("After oversampling: ",Counter(y_train))



## Building Model
- Logistic Regression
- Random Forest Classifier
- XGBoost Classifier
- Support vector machine classifier

### Logistic regression Model:

In [ ]:
from sklearn.linear_model import LogisticRegression
logit= LogisticRegression()
logit.fit(X_train, y_train)

In [ ]:
# Predicting the model
pred_logit= logit.predict(X_test)

#### Evaluation metrics

In [ ]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score, plot_confusion_matrix, plot_precision_recall_curve

print("The accuracy of logit model is:", accuracy_score(y_test, pred_logit))
print(classification_report(y_test, pred_logit))

In [ ]:
# Plot confusion metrics
plot_confusion_matrix(logit, X_test, y_test, cmap="Blues_r")

In [ ]:
# plot roc_auc curve
plot_precision_recall_curve(logit,X_test,y_test)

### Random Forest Classifer

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf= RandomForestClassifier()
# Fitting the model
rf.fit(X_train,y_train)

In [ ]:
# Predicting the model
pred_rf= rf.predict(X_test)

#### Evaluating the model

In [ ]:
print("The accuracy of logit model is:", accuracy_score(y_test, pred_rf))
print(classification_report(y_test,pred_rf ))

In [ ]:
# Plot confusion metrics
plot_confusion_matrix(rf, X_test, y_test, cmap="Blues_r")

In [ ]:
# plot pprcision_recall curve
plot_precision_recall_curve(rf,X_test,y_test)

### XGBoost Classifier

In [ ]:
import xgboost as xgb

xgb_clf= xgb.XGBClassifier()
#fitting the model
xgb_clf.fit(X_train,y_train)


In [ ]:
## Predicting the model
xgb_predict= xgb_clf.predict(X_test)

### Evaluating the model

In [ ]:
print("The accuracy of logit model is:", accuracy_score(y_test, xgb_predict))
print(classification_report(y_test,xgb_predict ))

In [ ]:
# Plot confusion metrics
plot_confusion_matrix(xgb_clf, X_test, y_test, cmap="Blues_r")

In [ ]:
# plot pprcision_recall curve
plot_precision_recall_curve(xgb_clf,X_test,y_test)

## Hyperparameter tunning

In [ ]:
## Hyper Parameter Optimization

params={
 "learning_rate"    : [0.05, 0.10, 0.15, 0.20, 0.25, 0.30 ] ,
 "max_depth"        : [ 3, 4, 5, 6, 8, 10, 12, 15],
 "min_child_weight" : [ 1, 3, 5, 7 ],
 "gamma"            : [ 0.0, 0.1, 0.2 , 0.3, 0.4 ],
 "colsample_bytree" : [ 0.3, 0.4, 0.5 , 0.7 ]
    
}

In [ ]:
## Hyperparameter optimization using RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV

In [ ]:
random_search=RandomizedSearchCV(xgb_clf,param_distributions=params,n_iter=5,scoring='roc_auc',n_jobs=-1,cv=5,verbose=3)

# fitting the RandomizedSearchCV
random_search.fit(X_train,y_train)

In [ ]:
# Finding the best estimators
random_search.best_estimator_

In [ ]:
# Finding the best param
random_search.best_params_

In [ ]:
classifier=xgboost.XGBClassifier(objective='binary:logistic',
                                  gamma=0.2,
                                  learning_rate=0.15,
                                  max_depth=15,
                                  reg_lambda=10,
                                  min_child_weight=1,
                                  subsample=0.9, 
                                  colsample_bytree=0.50,
                                  use_label_encoder=False)

# Fitting the model
classifier.fit(X_train,y_train)

In [ ]:
# Predicting model
y_pred= classifier.predict(X_test)

#### Evaluating the model after hyperparameter tuning

In [ ]:
print("The accuracy of the model is:", accuracy_score(y_test, y_pred))
print(classification_report(y_test,y_pred))

### Cross validation

In [ ]:
from sklearn.model_selection import cross_val_score
score=cross_val_score(classifier,X,y,cv=10)

In [ ]:
score

In [ ]:
score.mean()